CONNEXION + CREATION DE LA DB

In [15]:
# On importe le module sqlite3
# C'est une bibliothèque INCLUSE dans Python, pas besoin de l'installer
# Elle permet à Python de parler à une base de données SQLite
import sqlite3

# On importe le module os
# "os" signifie "operating system" - il permet à Python d'interagir 
# avec ton système de fichiers (créer des dossiers, vérifier si un fichier existe, etc.)
import os

# On crée le dossier "data/" sur ton ordinateur
# exist_ok=True signifie : "si le dossier existe déjà, ne fais rien"
# Sans exist_ok=True, Python lèverait une erreur si le dossier existe déjà
os.makedirs("data", exist_ok=True)

# On se connecte à la base de données SQLite
# sqlite3.connect() fait deux choses à la fois :
#   - Si le fichier "data/ventes.db" n'existe pas → il le CRÉE automatiquement
#   - Si le fichier existe déjà → il l'OUVRE simplement
# "conn" est la connexion — c'est le pont entre Python et ta BDD
conn = sqlite3.connect("data/ventes.db")

# Le cursor est le "stylo" qui va écrire et lire dans la BDD
# On ne parle jamais directement à conn pour exécuter du SQL
# On passe toujours par cursor.execute() pour envoyer des instructions SQL
cursor = conn.cursor()

print("connexion à ventes.db OK")



connexion à ventes.db OK


CREATION DES TABLES

In [16]:
# On envoie une série d'instructions SQL à SQLite pour créer les tables
# "CREATE TABLE IF NOT EXISTS" signifie :
#   - Si la table n'existe pas → on la CRÉE
#   - Si elle existe déjà → on ne fait RIEN (pas d'erreur, pas d'écrasement)
# C'est important car on va lancer ce script plusieurs fois

# --- TABLE MAGASINS ---
cursor.execute("""
    CREATE TABLE IF NOT EXISTS magasins (
        id_magasin VARCHAR PRIMARY KEY,
        ville VARCHAR,
        nombre_de_salaries INTEGER
    )
""")
# VARCHAR = texte de longueur variable (comme une chaîne de caractères)
# PRIMARY KEY = identifiant unique — il ne peut pas y avoir deux magasins avec le même id

# --- TABLE PRODUITS ---
cursor.execute("""
    CREATE TABLE IF NOT EXISTS produits (
        id_reference_produit VARCHAR PRIMARY KEY,
        nom VARCHAR,
        prix DECIMAL(10,2),
        stock INTEGER
    )
""")
# DECIMAL(10,2) = nombre avec 2 décimales — parfait pour un prix en euros (ex: 12.99)
# INTEGER = nombre entier — parfait pour un stock (ex: 150)

# --- TABLE VENTES ---
cursor.execute("""
    CREATE TABLE IF NOT EXISTS ventes (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        date DATETIME,
        id_reference_produit VARCHAR NOT NULL,
        quantite INTEGER,
        id_magasin VARCHAR NOT NULL,
        FOREIGN KEY (id_reference_produit) REFERENCES produits(id_reference_produit),
        FOREIGN KEY (id_magasin) REFERENCES magasins(id_magasin)
    )
""")
# AUTOINCREMENT = SQLite génère automatiquement un id unique à chaque nouvelle ligne
# NOT NULL = ce champ est obligatoire — on ne peut pas insérer une vente sans produit ou magasin
# FOREIGN KEY = clé étrangère — elle garantit qu'on ne peut pas insérer un id_magasin
#               qui n'existe pas dans la table magasins (même logique pour produits)

# --- TABLE RESULTATS ANALYSES ---
cursor.execute("""
    CREATE TABLE IF NOT EXISTS resultats_analyses (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        type_analyse VARCHAR NOT NULL,
        label VARCHAR,
        valeur DECIMAL(10,2),
        date_calcul DATETIME
    )
""")
# type_analyse = le nom de l'analyse : "ca_total", "par_produit", "par_region"
# label = la description du résultat : "Global", "Produit X", "Paris"
# valeur = le montant calculé en euros
# date_calcul = la date ET l'heure du calcul — pour garder un historique

# --- TABLE ERREUR IMPORT ---
cursor.execute("""
    CREATE TABLE IF NOT EXISTS erreurs_import (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        table_source VARCHAR,
        ligne INTEGER,
        colonne VARCHAR,
        valeur VARCHAR,
        raison VARCHAR,
        date_import DATETIME
    )
""")
# id = identifiant unique auto-généré par SQLite à chaque nouvelle erreur
# table_source = le nom du DataFrame concerné : "produits", "magasins" ou "ventes"
#                permet de savoir d'où vient l'erreur sans ouvrir les fichiers
# ligne = le numéro de la ligne dans le CSV (index pandas)
#         permet de retrouver exactement la ligne problématique dans le fichier source
# colonne = le nom de la colonne qui contient la valeur incorrecte
#           ex: "quantite", "prix", "id_reference_produit"
# valeur = la valeur incorrecte elle-même, stockée en texte
#          on utilise VARCHAR car la valeur peut être de n'importe quel type
# raison = l'explication de pourquoi cette valeur est invalide
#          ex: "quantite <= 0", "id_magasin inexistant dans magasins"
# date_import = la date ET l'heure de la tentative d'import
#               permet de savoir à quel run l'erreur a été détectée

# conn.commit() sauvegarde toutes les modifications dans le fichier ventes.db
# Sans commit(), les tables sont créées en mémoire mais pas sauvegardées sur disque
conn.commit()

print("✅ Tables créées avec succès")

✅ Tables créées avec succès


TELECHARGEMENT DES DONNEES

In [17]:
# On importe requests — bibliothèque pour faire des requêtes HTTP
# C'est elle qui va "appeler" les URLs comme un navigateur web le ferait
# requests n'est PAS incluse dans Python, il faudra l'ajouter dans requirements.txt
import requests

# On importe pandas — bibliothèque pour manipuler des tableaux de données
# pandas lit le CSV et le transforme en DataFrame (tableau Python manipulable)
# pandas n'est PAS incluse dans Python non plus → requirements.txt
import pandas as pd

# On importe StringIO — outil qui transforme du texte brut en "faux fichier"
# pandas attend un fichier pour lire un CSV
# StringIO lui fait croire que le texte téléchargé EST un fichier
from io import StringIO

# Les URLs des 3 CSV fournis par le client
# Tu remplaceras ces URLs par celles que Simplon t'a données
URL_PRODUITS  = "https://docs.google.com/spreadsheets/d/e/2PACX-1vSawI56WBC64foMT9pKCiY594fBZk9Lyj8_bxfgmq-8ck_jw1Z49qDeMatCWqBxehEVoM6U1zdYx73V/pub?gid=0&single=true&output=csv"
URL_MAGASINS  = "https://docs.google.com/spreadsheets/d/e/2PACX-1vSawI56WBC64foMT9pKCiY594fBZk9Lyj8_bxfgmq-8ck_jw1Z49qDeMatCWqBxehEVoM6U1zdYx73V/pub?gid=714623615&single=true&output=csv"
URL_VENTES    = "https://docs.google.com/spreadsheets/d/e/2PACX-1vSawI56WBC64foMT9pKCiY594fBZk9Lyj8_bxfgmq-8ck_jw1Z49qDeMatCWqBxehEVoM6U1zdYx73V/pub?gid=760830694&single=true&output=csv"

# --- TÉLÉCHARGEMENT CSV PRODUITS ---
# requests.get() envoie une requête HTTP GET vers l'URL
# La réponse contient le contenu brut du fichier CSV dans response.text
response_produits = requests.get(URL_PRODUITS)

# On transforme le texte brut en DataFrame pandas
# StringIO(response_produits.text) = on fait croire à pandas que c'est un fichier
df_produits = pd.read_csv(StringIO(response_produits.content.decode('utf-8')))

# On affiche les 5 premières lignes pour vérifier que c'est correct
print("✅ Produits téléchargés :", df_produits.shape)
display(df_produits.head())

# --- TÉLÉCHARGEMENT CSV MAGASINS ---
response_magasins = requests.get(URL_MAGASINS)
df_magasins = pd.read_csv(StringIO(response_magasins.content.decode('utf-8')))
print("✅ Magasins téléchargés :", df_magasins.shape)
display(df_magasins.head())

# --- TÉLÉCHARGEMENT CSV VENTES ---
response_ventes = requests.get(URL_VENTES)
df_ventes = pd.read_csv(StringIO(response_ventes.content.decode('utf-8')))
print("✅ Ventes téléchargées :", df_ventes.shape)
display(df_ventes.head())

✅ Produits téléchargés : (5, 4)


,Nom,ID Référence produit,Prix,Stock
0,Produit A,REF001,49.99,100
1,Produit B,REF002,19.99,50
2,Produit C,REF003,29.99,75
3,Produit D,REF004,79.99,120
4,Produit E,REF005,39.99,80


✅ Magasins téléchargés : (7, 3)


,ID Magasin,Ville,Nombre de salariés
0,1,Paris,10
1,2,Marseille,5
2,3,Lyon,8
3,4,Bordeaux,12
4,5,Lille,6


✅ Ventes téléchargées : (30, 4)


,Date,ID Référence produit,Quantité,ID Magasin
0,2023-05-27,REF001,5,1
1,2023-05-28,REF002,3,2
2,2023-05-29,REF003,2,1
3,2023-05-30,REF004,4,3
4,2023-05-31,REF005,7,2


In [18]:
print(df_ventes.columns.tolist())

['Date', 'ID Référence produit', 'Quantité', 'ID Magasin']


In [4]:
df_produits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Nom                   5 non-null      object 
 1   ID Référence produit  5 non-null      object 
 2   Prix                  5 non-null      float64
 3   Stock                 5 non-null      int64  
dtypes: float64(1), int64(1), object(2)
memory usage: 292.0+ bytes


In [5]:
df_magasins.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   ID Magasin          7 non-null      int64 
 1   Ville               7 non-null      object
 2   Nombre de salariés  7 non-null      int64 
dtypes: int64(2), object(1)
memory usage: 300.0+ bytes


In [6]:
df_ventes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Date                  30 non-null     object
 1   ID Référence produit  30 non-null     object
 2   Quantité              30 non-null     int64 
 3   ID Magasin            30 non-null     int64 
dtypes: int64(2), object(2)
memory usage: 1.1+ KB


NETTOYAGE
clean / rename ...

In [7]:
df_produits.head()

,Nom,ID Référence produit,Prix,Stock
0,Produit A,REF001,49.99,100
1,Produit B,REF002,19.99,50
2,Produit C,REF003,29.99,75
3,Produit D,REF004,79.99,120
4,Produit E,REF005,39.99,80


In [ ]:
# --- NETTOYAGE df_produits ---
# rename() remplace les noms de colonnes du CSV par ceux de notre BDD
# "columns" prend un dictionnaire : {"ancien nom" : "nouveau nom"}
df_produits = df_produits.rename(columns={
    "Nom" : "nom",
    "ID Référence produit" : "id_reference_produit",
    "Prix" : "prix",
    "Stock" : "stock"
})

# --- NETTOYAGE df_magasins ---
df_magasins = df_magasins.rename(columns={
    "ID Magasin" : "id_magasin", 
    "Ville" : "ville",
    "Nombre de salariés" : "nombre_de_salaries"
})

# --- NETTOYAGE df_ventes ---
df_ventes = df_ventes.rename(columns={
    "Date" : "date",
    "ID Référence produit" : "id_reference_produit",
    "Quantité" : "quantite",
    "ID Magasin" : "id_magasin"
})

In [9]:
# On verifie que les noms sont corrects
print("colonnes produits :", df_produits.columns.tolist())
print("colonnes magasins :", df_magasins.columns.tolist())
print("colonnes ventes :", df_ventes.columns.tolist())

colonnes produits : ['nom', 'id_reference_produit', 'prix', 'stock']
colonnes magasins : ['id_magasin', 'ville', 'nombre_de_salaries']
colonnes ventes : ['date', 'id_reference_produit', 'quantite', 'id_magasin']


In [10]:
# On vérifie s'il y a des valeurs manquantes dans chaque DataFrame
print("=== PRODUITS ===")
print(df_produits.isnull().sum())

print("\n=== MAGASINS ===")
print(df_magasins.isnull().sum())

print("\n=== VENTES ===")
print(df_ventes.isnull().sum())


=== PRODUITS ===
nom                     0
id_reference_produit    0
prix                    0
stock                   0
dtype: int64

=== MAGASINS ===
id_magasin            0
ville                 0
nombre_de_salaries    0
dtype: int64

=== VENTES ===
date                    0
id_reference_produit    0
quantite                0
id_magasin              0
dtype: int64


In [11]:
# On vérifie les types de données
print("\n=== TYPES PRODUITS ===")
print(df_produits.dtypes)

print("\n=== TYPES MAGASINS ===")
print(df_magasins.dtypes)

print("\n=== TYPES VENTES ===")
print(df_ventes.dtypes)


=== TYPES PRODUITS ===
nom                      object
id_reference_produit     object
prix                    float64
stock                     int64
dtype: object

=== TYPES MAGASINS ===
id_magasin             int64
ville                 object
nombre_de_salaries     int64
dtype: object

=== TYPES VENTES ===
date                    object
id_reference_produit    object
quantite                 int64
id_magasin               int64
dtype: object


In [12]:
# La colonne date est actuellement du texte (object)
# On la convertit en vrai datetime pour pouvoir faire des calculs temporels
# pd.to_datetime() reconnaît automatiquement le format de date
df_ventes["date"] = pd.to_datetime(df_ventes["date"])

In [13]:
# On vérifie
print("\n=== TYPES VENTES ===")
print(df_ventes.dtypes)


=== TYPES VENTES ===
date                    datetime64[ns]
id_reference_produit            object
quantite                         int64
id_magasin                       int64
dtype: object


In [14]:
from datetime import datetime

# ============================================
# VALIDATION DES DONNÉES AVANT IMPORT
# ============================================

# Fonction qui enregistre une erreur dans la table erreurs_import
# plutôt que de juste afficher un message et s'arrêter
def log_erreur(table_source, ligne, colonne, valeur, raison):
    cursor.execute("""
        INSERT INTO erreurs_import 
        (table_source, ligne, colonne, valeur, raison, date_import)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        table_source,                              # nom du DataFrame
        ligne,                                     # numéro de la ligne
        colonne,                                   # nom de la colonne
        str(valeur),                               # valeur incorrecte en texte
        raison,                                    # explication de l'erreur
        datetime.now().strftime("%Y-%m-%d %H:%M:%S")  # date et heure du run
    ))


# On va stocker les lignes VALIDES dans de nouveaux DataFrames
# Les lignes invalides seront envoyées dans erreurs_import

# --- VALIDATION PRODUITS ---
produits_inseres = 0
produits_ignores = 0

for index, row in df_produits.iterrows():

    # 1. Vérifications anticipées
    if row['prix'] <= 0:
        log_erreur("produits", index, "prix", row['prix'], "prix <= 0")
        continue  # on passe à la ligne suivante sans insérer

    if row['stock'] < 0:
        log_erreur("produits", index, "stock", row['stock'], "stock négatif")
        continue

    # 2. Déduplication
    cursor.execute(
        "SELECT id_reference_produit FROM produits WHERE id_reference_produit = ?",
        (row['id_reference_produit'],)
    )
    if cursor.fetchone() is not None:
        produits_ignores += 1
        continue  # déjà présent → on passe à la ligne suivante

    # 3. Insertion avec capture des erreurs inattendues
    try:
        cursor.execute("""
            INSERT INTO produits (id_reference_produit, nom, prix, stock)
            VALUES (?, ?, ?, ?)
        """, (row['id_reference_produit'], row['nom'], row['prix'], row['stock']))
        produits_inseres += 1

    except Exception as e:
        log_erreur("produits", index, "inconnu", str(row.to_dict()), str(e))
        produits_ignores += 1

print(f"✅ Produits — {produits_inseres} insérés / {produits_ignores} ignorés")


# --- VALIDATION MAGASINS ---
magasins_valides = []

for index, row in df_magasins.iterrows():

    ligne_valide = True

    # Vérification : nombre de salariés doit être supérieur à 0
    if row['nombre_de_salaries'] <= 0:
        log_erreur("magasins", index, "nombre_de_salaries", row['nombre_de_salaries'], "nombre_de_salaries <= 0")
        ligne_valide = False

    if ligne_valide:
        magasins_valides.append(row)

df_magasins_valides = pd.DataFrame(magasins_valides)
print(f"✅ Magasins — {len(df_magasins_valides)}/{len(df_magasins)} lignes valides")

# --- VALIDATION VENTES ---
ventes_valides = []

# On prépare des sets pour vérifier l'intégrité référentielle
# Un set c'est une liste sans doublons — très rapide pour vérifier si une valeur existe
refs_produits_valides = set(df_produits_valides['id_reference_produit'])
ids_magasins_valides  = set(df_magasins_valides['id_magasin'].astype(str))

for index, row in df_ventes.iterrows():

    ligne_valide = True

    # Vérification : la quantité doit être supérieure à 0
    if row['quantite'] <= 0:
        log_erreur("ventes", index, "quantite", row['quantite'], "quantite <= 0")
        ligne_valide = False

    # Vérification : le produit référencé doit exister dans produits
    if row['id_reference_produit'] not in refs_produits_valides:
        log_erreur("ventes", index, "id_reference_produit", row['id_reference_produit'], "id_reference_produit inexistant dans produits")
        ligne_valide = False

    # Vérification : le magasin référencé doit exister dans magasins
    if str(row['id_magasin']) not in ids_magasins_valides:
        log_erreur("ventes", index, "id_magasin", row['id_magasin'], "id_magasin inexistant dans magasins")
        ligne_valide = False

    if ligne_valide:
        ventes_valides.append(row)

df_ventes_valides = pd.DataFrame(ventes_valides)
print(f"✅ Ventes — {len(df_ventes_valides)}/{len(df_ventes)} lignes valides")

# On sauvegarde les erreurs dans la BDD
conn.commit()

# On affiche un résumé des erreurs détectées
cursor.execute("SELECT COUNT(*) FROM erreurs_import")
nb_erreurs = cursor.fetchone()[0]
if nb_erreurs > 0:
    print(f"\n⚠️ {nb_erreurs} erreur(s) enregistrée(s) dans erreurs_import")
else:
    print("\n✅ Aucune erreur détectée — toutes les données sont valides")

✅ Produits — 0 insérés / 5 ignorés
✅ Magasins — 7/7 lignes valides


NameError: name 'df_produits_valides' is not defined

In [ ]:
# On vérifie s'il y a des espaces avant ou après dans les colonnes texte
print(df_produits_valides['nom'].tolist())
print(df_produits_valides['id_reference_produit'].tolist())
print(df_magasins_valides['ville'].tolist())
print(df_ventes_valides['id_reference_produit'].tolist())

['Produit A', 'Produit B', 'Produit C', 'Produit D', 'Produit E']
['REF001', 'REF002', 'REF003', 'REF004', 'REF005']
['Paris', 'Marseille', 'Lyon', 'Bordeaux', 'Lille', 'Nantes', 'Strasbourg']
['REF001', 'REF002', 'REF003', 'REF004', 'REF005', 'REF001', 'REF002', 'REF003', 'REF004', 'REF005', 'REF001', 'REF002', 'REF003', 'REF004', 'REF005', 'REF001', 'REF002', 'REF003', 'REF004', 'REF005', 'REF001', 'REF002', 'REF003', 'REF004', 'REF005', 'REF001', 'REF002', 'REF003', 'REF004', 'REF005']


In [ ]:
# ============================================
# NETTOYAGE FINAL AVANT IMPORT
# ============================================

# --- PRODUITS ---
# strip() supprime les espaces avant et après chaque valeur texte
# upper() met tout en majuscule pour éviter les doublons type "Produit A" vs "produit a"
df_produits_valides['nom'] = df_produits_valides['nom'].str.strip().str.upper()
df_produits_valides['id_reference_produit'] = df_produits_valides['id_reference_produit'].str.strip().str.upper()

# --- MAGASINS ---
# strip() sur toutes les colonnes texte
# upper() sur ville pour éviter "Paris" vs "paris" vs "PARIS"
df_magasins_valides['ville'] = df_magasins_valides['ville'].str.strip().str.upper()
df_magasins_valides['id_magasin'] = df_magasins_valides['id_magasin'].astype(str).str.strip()

# --- VENTES ---
# strip() sur les colonnes texte
df_ventes_valides['id_reference_produit'] = df_ventes_valides['id_reference_produit'].str.strip().str.upper()
df_ventes_valides['id_magasin'] = df_ventes_valides['id_magasin'].astype(str).str.strip()

# On vérifie le résultat
print("=== PRODUITS ===")
print(df_produits_valides[['nom', 'id_reference_produit']].values.tolist())
display(df_produits_valides.head())

print("=== MAGASINS ===")
display(df_magasins_valides.head())

print("=== VENTES ===")
display(df_ventes_valides.head())

=== PRODUITS ===
[['PRODUIT A', 'REF001'], ['PRODUIT B', 'REF002'], ['PRODUIT C', 'REF003'], ['PRODUIT D', 'REF004'], ['PRODUIT E', 'REF005']]


,nom,id_reference_produit,prix,stock
0,PRODUIT A,REF001,49.99,100
1,PRODUIT B,REF002,19.99,50
2,PRODUIT C,REF003,29.99,75
3,PRODUIT D,REF004,79.99,120
4,PRODUIT E,REF005,39.99,80


=== MAGASINS ===


,id_magasin,ville,nombre_de_salaries
0,1,PARIS,10
1,2,MARSEILLE,5
2,3,LYON,8
3,4,BORDEAUX,12
4,5,LILLE,6


=== VENTES ===


,date,id_reference_produit,quantite,id_magasin
0,2023-05-27,REF001,5,1
1,2023-05-28,REF002,3,2
2,2023-05-29,REF003,2,1
3,2023-05-30,REF004,4,3
4,2023-05-31,REF005,7,2


IMPORT DES DONNÉES DANS LA BDD

In [ ]:
# --- IMPORT PRODUITS ---
# On parcourt chaque ligne du DataFrame valide et nettoyé
produits_inseres = 0
produits_ignores = 0

for index, row in df_produits_valides.iterrows():
    
    # On vérifie si ce produit existe déjà dans la BDD
    # via son id_reference_produit qui est la clé primaire
    cursor.execute(
        "SELECT id_reference_produit FROM produits WHERE id_reference_produit = ?",
        (row['id_reference_produit'],)
    )
    
    # fetchone() retourne la ligne trouvée ou None si rien
    existe = cursor.fetchone()
    
    if existe is None:
        # Le produit n'existe pas encore → on l'insère
        cursor.execute("""
            INSERT INTO produits (id_reference_produit, nom, prix, stock)
            VALUES (?, ?, ?, ?)
        """, (
            row['id_reference_produit'],
            row['nom'],
            row['prix'],
            row['stock']
        ))
        produits_inseres += 1
    else:
        # Le produit existe déjà → on ne fait rien
        produits_ignores += 1

print(f"✅ Produits — {produits_inseres} insérés / {produits_ignores} ignorés (déjà présents)")

# --- IMPORT MAGASINS ---
magasins_inseres = 0
magasins_ignores = 0

for index, row in df_magasins_valides.iterrows():
    
    cursor.execute(
        "SELECT id_magasin FROM magasins WHERE id_magasin = ?",
        (str(row['id_magasin']),)
    )
    existe = cursor.fetchone()
    
    if existe is None:
        cursor.execute("""
            INSERT INTO magasins (id_magasin, ville, nombre_de_salaries)
            VALUES (?, ?, ?)
        """, (
            str(row['id_magasin']),
            row['ville'],
            row['nombre_de_salaries']
        ))
        magasins_inseres += 1
    else:
        magasins_ignores += 1

print(f"✅ Magasins — {magasins_inseres} insérés / {magasins_ignores} ignorés (déjà présents)")

# --- IMPORT VENTES ---
# C'est la table la plus importante pour la déduplication
# car le fichier ventes s'actualise en temps réel
ventes_inserees = 0
ventes_ignorees = 0

for index, row in df_ventes_valides.iterrows():
    
    # Pour les ventes on n'a pas d'id unique dans le CSV
    # On déduplique sur la combinaison date + produit + magasin + quantite
    # Si ces 4 valeurs sont identiques → c'est la même vente
    cursor.execute("""
        SELECT id FROM ventes 
        WHERE date = ? 
        AND id_reference_produit = ? 
        AND id_magasin = ?
        AND quantite = ?
    """, (
        str(row['date']),
        row['id_reference_produit'],
        str(row['id_magasin']),
        row['quantite']
    ))
    existe = cursor.fetchone()
    
    if existe is None:
        cursor.execute("""
            INSERT INTO ventes (date, id_reference_produit, quantite, id_magasin)
            VALUES (?, ?, ?, ?)
        """, (
            str(row['date']),
            row['id_reference_produit'],
            row['quantite'],
            str(row['id_magasin'])
        ))
        ventes_inserees += 1
    else:
        ventes_ignorees += 1

# On sauvegarde tout dans le fichier ventes.db
conn.commit()

print(f"✅ Ventes — {ventes_inserees} insérées / {ventes_ignorees} ignorées (déjà présentes)")
print("\n✅ Import terminé — vérifie dans DB Browser !")

✅ Produits — 0 insérés / 5 ignorés (déjà présents)
✅ Magasins — 0 insérés / 7 ignorés (déjà présents)
✅ Ventes — 0 insérées / 30 ignorées (déjà présentes)

✅ Import terminé — vérifie dans DB Browser !


In [ ]:
# ============================================
# EXÉCUTION DES ANALYSES ET STOCKAGE DES RÉSULTATS
# ============================================

from datetime import datetime

# On lit le fichier analyses.sql depuis le disque
# "r" = read, "utf-8" = encodage du fichier
with open("analyses.sql", "r", encoding="utf-8") as f:
    sql_brut = f.read()

# On sépare les 3 requêtes en les découpant sur ";"
# strip() supprime les espaces et sauts de ligne autour de chaque requête
# filter(None, ...) supprime les éléments vides
requetes = [r.strip() for r in sql_brut.split(";") if r.strip()]

# La date du calcul sera la même pour tout ce run
date_calcul = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# ============================================
# a) CHIFFRE D'AFFAIRES TOTAL
# ============================================
cursor.execute(requetes[0])
# fetchone() récupère une seule ligne — normal car SUM() retourne 1 résultat
resultat_ca = cursor.fetchone()
ca_total = resultat_ca[0]

# On stocke le résultat dans resultats_analyses
cursor.execute("""
    INSERT INTO resultats_analyses (type_analyse, label, valeur, date_calcul)
    VALUES (?, ?, ?, ?)
""", ("ca_total", "Chiffre d'affaires global", ca_total, date_calcul))

print(f"✅ CA total : {ca_total} €")

# ============================================
# b) VENTES PAR PRODUIT
# ============================================
cursor.execute(requetes[1])
# fetchall() récupère toutes les lignes — car on a plusieurs produits
resultats_produits = cursor.fetchall()

for row in resultats_produits:
    # row[0] = nom produit, row[1] = quantité, row[2] = CA produit
    cursor.execute("""
        INSERT INTO resultats_analyses (type_analyse, label, valeur, date_calcul)
        VALUES (?, ?, ?, ?)
    """, ("par_produit", row[0], row[2], date_calcul))
    print(f"  → {row[0]} : {row[2]} €")

print("✅ Ventes par produit stockées")

# ============================================
# c) VENTES PAR RÉGION
# ============================================
cursor.execute(requetes[2])
resultats_regions = cursor.fetchall()

for row in resultats_regions:
    # row[0] = ville, row[1] = quantité, row[2] = CA région
    cursor.execute("""
        INSERT INTO resultats_analyses (type_analyse, label, valeur, date_calcul)
        VALUES (?, ?, ?, ?)
    """, ("par_region", row[0], row[2], date_calcul))
    print(f"  → {row[0]} : {row[2]} €")

print("✅ Ventes par région stockées")

# On sauvegarde tout dans ventes.db
conn.commit()

print("\n✅ Tous les résultats sont stockés dans resultats_analyses")

✅ CA total : 5268.78 €
  → PRODUIT D : 1679.79 €
  → PRODUIT E : 1399.65 €
  → PRODUIT A : 1199.76 €
  → PRODUIT B : 539.73 €
  → PRODUIT C : 449.85 €
✅ Ventes par produit stockées
  → LYON : 1059.79 €
  → MARSEILLE : 1009.73 €
  → BORDEAUX : 829.81 €
  → PARIS : 799.8 €
  → NANTES : 739.83 €
  → STRASBOURG : 579.89 €
  → LILLE : 249.93 €
✅ Ventes par région stockées

✅ Tous les résultats sont stockés dans resultats_analyses
